# Diagnose and hand off a small source project

**Goal:** reconcile a saved methods note, Python source, and saved notebook exercise. This fabricated, disposable fixture says values **at or above** a threshold qualify, while its initial Python code uses `>`. You will locate the discrepancy, make a SHA-256-checked repair, run a bounded fresh-process test, and save a handoff report.

Run Python cells in order for a complete provider-free route. The AI questions are optional: to try them, connect a provider in **Configure AI** and run them separately. `&` declarations offer tools; `$` references supply current kernel paths. AI answers do not assign Python variables. The final code checkpoint checks the saved source and test before writing a report in `nbinlineai-results/project-…` under the kernel's current directory.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
import hashlib
import json
import shutil
from nbinlineai.tools import (
    search_files, source_doc, document_outline, read_document_section,
    search_notebooks, notebook_outline, read_notebook_cell,
    view_file_hashes, file_replace_checked, run_python,
)

project_scratch = TemporaryDirectory(prefix="nbinlineai-handoff-")
project_root = Path(project_scratch.name)
source_path = project_root / "counts.py"
notes_path = project_root / "METHODS.md"
saved_notebook_path = project_root / "lesson.ipynb"
source_path.write_text('"""Count readings that meet a stated threshold."""\n\ndef count_qualifying(values: list[int], threshold: int) -> int:\n    """Return how many readings qualify."""\n    return sum(value > threshold for value in values)\n', encoding="utf-8")
notes_path.write_text('# Counting method\n\n## Inclusion rule\nA reading qualifies when it is **at or above** the threshold. For `[2, 3, 4]` and threshold `3`, the answer is `2`.\n', encoding="utf-8")
saved_notebook_path.write_text(json.dumps({
    "cells": [{"cell_type": "markdown", "id": "lesson-rule", "metadata": {}, "source": "# Exercise\nCheck the inclusion rule in METHODS.md."},
              {"cell_type": "code", "id": "lesson-check", "metadata": {}, "execution_count": None, "outputs": [],
               "source": "from counts import count_qualifying\nprint(count_qualifying([2, 3, 4], 3))"}],
    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}},
    "nbformat": 4, "nbformat_minor": 5,
}), encoding="utf-8")
print("Disposable project:", project_root)
print("Source:", source_path.name, "Methods:", notes_path.name, "Saved lesson:", saved_notebook_path.name)


## Read the three saved sources

These ordinary Python calls inspect saved files, including a saved notebook cell. A saved `.ipynb` read cannot see unsaved edits in an open tab. Section addresses from `document_outline` become stale after that file changes; request a fresh outline before using one.


In [ ]:
print(search_files("count_qualifying", path=str(project_root), pattern="*.py"))
print(source_doc(str(source_path), symbol="count_qualifying"))
print(view_file_hashes(str(source_path), start_line=1, end_line=5))
print(document_outline(str(notes_path)))
print(read_document_section(str(notes_path)))
print(search_notebooks("count_qualifying", path=str(project_root)))
print(notebook_outline(str(saved_notebook_path)))
print(read_notebook_cell(str(saved_notebook_path), "lesson-check"))


## Tools offered to lower questions

- &`search_files` — locate the saved implementation.
- &`source_doc` — read its definition without importing it.
- &`document_outline` — list methods headings and addresses.
- &`read_document_section` — read the saved methods rule.
- &`search_notebooks` — find saved notebook cells by source text.
- &`read_notebook_cell` — read one saved cell by ID.
- &`view_file_hashes` — get numbered lines and a whole-file SHA-256.
- &`file_replace_checked` — replace one literal only when the digest still matches.
- &`run_python` — test in a bounded fresh process, separate from this kernel's variables.

Keep this ordinary Markdown cell's **Tools** checkbox on. The kernel reads and writes these paths with its filesystem permissions. Use only the fixture above; do not edit the saved `.ipynb` with text-edit tools.


In the disposable project $`project_root`, use `search_files` to locate `count_qualifying`, `source_doc` for its signature, and `view_file_hashes` to read the saved function body. Read the methods rule, then find/read saved `lesson-check` in $`saved_notebook_path`. Explain the exact difference for `[2, 3, 4]` at threshold `3`. Do not edit; identify the source literal that needs repair.


## Checkpoint before the repair

The next direct calls show the current whole-file digest and run a boundary test. Failure is expected **before** the edit. If you already completed the repair, the same test will pass. A printed AI answer alone does not establish that a file changed.


In [ ]:
print(view_file_hashes(str(source_path), start_line=1, end_line=5))
boundary_test_code = "from counts import count_qualifying; assert count_qualifying([2, 3, 4], 3) == 2; print('boundary test passed')"
print(run_python(boundary_test_code, cwd=str(project_root), timeout=5))


Optional AI route: repair only $`source_path`. The following Python cell also provides a checked repair and safely skips an already repaired file. First call `view_file_hashes` for its **current** whole-file SHA-256. Then call `file_replace_checked` with that digest, changing the unique literal `value > threshold` to `value >= threshold`. Do not modify the methods note or saved notebook. Finally call `run_python` in $`project_root` with this fresh-process test: `from counts import count_qualifying; assert count_qualifying([2, 3, 4], 3) == 2; print('boundary test passed')`. Report the actual edit and exit status. If the hash or literal check fails, stop and explain; do not guess or retry with a stale digest.


## Direct checked repair (works without AI)

Run the next Python cell whether or not you ran the optional AI repair question. It reads the current source and SHA-256, checks the exact expression, and uses `file_replace_checked` only while the original expression is present. If the AI already repaired it, the cell reports a safe skip. It then runs the boundary test in a fresh process. If the source changed in another way, it stops for inspection.


In [ ]:
old_expression = "return sum(value > threshold for value in values)"
new_expression = "return sum(value >= threshold for value in values)"
current_source = source_path.read_text(encoding="utf-8")
print(view_file_hashes(str(source_path), start_line=1, end_line=5))
if current_source.count(new_expression) == 1 and old_expression not in current_source:
    print("Checked repair already present; skipped the edit.")
else:
    assert current_source.count(old_expression) == 1 and new_expression not in current_source, (
        "Unexpected source change; inspect counts.py before repairing."
    )
    current_sha256 = hashlib.sha256(current_source.encode("utf-8")).hexdigest()
    print(file_replace_checked(
        str(source_path), old_str=old_expression, new_str=new_expression,
        expected_sha256=current_sha256,
    ))
checked_test = run_python(boundary_test_code, cwd=str(project_root), timeout=5)
assert checked_test.startswith("Exit code: 0.") and "boundary test passed" in checked_test, checked_test
print(checked_test)


## Verify and write the handoff

Run the next cell after the direct repair. It checks the saved source and reruns the bounded test before writing a report. It also copies repaired `counts.py`, `METHODS.md`, and the saved `lesson.ipynb` beside the report, so cleanup of the disposable fixture leaves a usable project. If a check fails, the cell stops without claiming completion. Re-running a successful checkpoint creates a new report folder.


In [ ]:
current_source = source_path.read_text(encoding="utf-8")
assert "value >= threshold" in current_source and "value > threshold" not in current_source, (
    "Source still has the mismatch. Inspect it and complete the checked repair first."
)
verified_test = run_python(boundary_test_code, cwd=str(project_root), timeout=5)
assert verified_test.startswith("Exit code: 0."), verified_test
assert "boundary test passed" in verified_test, verified_test
report_dir = Path.cwd() / "nbinlineai-results" / f"project-{uuid4().hex[:8]}"
report_dir.mkdir(parents=True, exist_ok=False)
for original in (source_path, notes_path, saved_notebook_path):
    shutil.copy2(original, report_dir / original.name)
exported_source = report_dir / source_path.name
source_sha256 = hashlib.sha256(exported_source.read_bytes()).hexdigest()
assert source_sha256 == hashlib.sha256(source_path.read_bytes()).hexdigest()
handoff_path = report_dir / "handoff.md"
handoff = f"""# Counting method handoff

Fixture: fabricated disposable project at `{project_root}`. The repaired source, methods note and saved lesson are copied beside this report and survive fixture cleanup.

## Evidence
- [METHODS.md](METHODS.md) rule: a value at or above the threshold qualifies.
- [lesson.ipynb](lesson.ipynb), saved cell `lesson-check`: calls `count_qualifying([2, 3, 4], 3)`.
- [counts.py](counts.py) originally used `value > threshold`, excluding the boundary value 3.
- Repair: saved source now uses `value >= threshold`.
- Exported `counts.py` SHA-256: `{source_sha256}`.

## Check
Fresh Python process in the fixture directory: `{verified_test.splitlines()[0]}`. Its output includes `boundary test passed`; expected count is 2.

## Follow-up
Extend tests for empty input, repeated threshold values and real data before reusing the method. An AI answer itself is not a source or test result; this report records saved source and a fresh-process check.
"""
handoff_path.write_text(handoff, encoding="utf-8")
print(handoff)
print("Saved handoff:", handoff_path)
print("Copied project files:", source_path.name, notes_path.name, saved_notebook_path.name)


## Keep or clean up

Open the printed `handoff_path` to review it. The fixture exists while `project_scratch` is alive. When you no longer need it, run `project_scratch.cleanup()` in a new code cell; the handoff and copied project files remain. If a model action times out, inspect the file and digest before retrying to avoid a second edit.
